# Data Preparation and Consolidation — Slovakia

## Objective

The objective of this stage is to inspect, clean, and consolidate the Slovak bilateral trade dataset into a unified monthly dataset suitable for subsequent exploratory data analysis and forecasting.

The processing workflow focuses on:

- Verifying the structure and temporal coverage of the source data.
- Assessing data completeness and consistency.
- Identifying and investigating duplicate observations.
- Establishing an appropriate procedure for handling duplicate records.
- Consolidating observations at the intended analytical level.
- Exporting the processed dataset in Parquet format.
- Validating the resulting dataset.

The original source file will be retained without modification.

## 1. Imports and configurations

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np

In [2]:
PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "Data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "Data" / "processed"

print(f"Project directory: {PROJECT_DIR}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")

assert RAW_DIR.exists(), "Raw data directory does not exist."
assert PROCESSED_DIR.exists(), "Processed data directory does not exist."

Project directory: c:\Transfer\InterdisciplinaryProject
Raw data directory: c:\Transfer\InterdisciplinaryProject\data\raw
Processed data directory: c:\Transfer\InterdisciplinaryProject\data\processed


## 2. Data Source and Input File

The Slovak bilateral trade dataset was obtained from Eurostat and is provided as a single CSV file containing monthly trade observations at the product and trading-partner levels.

The source file is located in `data/raw/` and will be retained unchanged throughout the preprocessing workflow.

In [3]:
SLOVAKIA_FILE = RAW_DIR / "trade_n_svk_m_hs.csv"

assert SLOVAKIA_FILE.exists(), f"Source file not found: {SLOVAKIA_FILE}"

file_size_gb = SLOVAKIA_FILE.stat().st_size / (1024 ** 3)

print(f"Source file: {SLOVAKIA_FILE.name}")
print(f"File size: {file_size_gb:.2f} GB")

Source file: trade_n_svk_m_hs.csv
File size: 1.81 GB


### 2.1 Inspect the source schema
The source schema is inspected to identify the available variables and their inferred data types before conducting the data quality assessment.

In [4]:
con = duckdb.connect(database=":memory:")

schema = con.execute(
    "DESCRIBE SELECT * FROM read_csv_auto(?)",
    [str(SLOVAKIA_FILE)]
).df()

schema

,column_name,column_type,null,key,default,extra
0,month_id,BIGINT,YES,None,None,None
1,product_id,VARCHAR,YES,None,None,None
2,product_name,VARCHAR,YES,None,None,None
3,country_id,VARCHAR,YES,None,None,None
4,country_name,VARCHAR,YES,None,None,None
5,trade_flow_name,VARCHAR,YES,None,None,None
6,trade_value,BIGINT,YES,None,None,None


## 3. Data Inspection and Quality Assessment

The Slovak trade dataset is examined to assess its temporal coverage, data completeness, and structural consistency before investigating duplicate observations and preparing the data for consolidation.

### 3.1 Temporal Coverage

The temporal coverage of the dataset is examined by identifying the first and last available months and counting the number of distinct monthly observations.

In [5]:
temporal_coverage = con.execute("""
    SELECT
        MIN(month_id) AS first_month,
        MAX(month_id) AS last_month,
        COUNT(DISTINCT month_id) AS distinct_months
    FROM read_csv_auto(?)
""", [str(SLOVAKIA_FILE)]).df()

temporal_coverage

,first_month,last_month,distinct_months
0,201501,202603,135


### 3.2 Temporal Consistency Check

The observed monthly identifiers are compared against the expected sequence of consecutive months to identify any missing or unexpected reporting periods.

In [6]:
temporal_check = con.execute("""
    WITH observed_months AS (
        SELECT DISTINCT month_id
        FROM read_csv_auto(?)
    ),
    expected_months AS (
        SELECT
            year * 100 + month AS month_id
        FROM range(2015, 2027) AS years(year)
        CROSS JOIN range(1, 13) AS months(month)
        WHERE year * 100 + month BETWEEN 201501 AND 202603
    )
    SELECT
        (SELECT COUNT(*) FROM observed_months) AS observed_months,
        (SELECT COUNT(*) FROM expected_months) AS expected_months,
        (SELECT COUNT(*) FROM expected_months e
         LEFT JOIN observed_months o USING (month_id)
         WHERE o.month_id IS NULL) AS missing_months,
        (SELECT COUNT(*) FROM observed_months o
         LEFT JOIN expected_months e USING (month_id)
         WHERE e.month_id IS NULL) AS unexpected_months
""", [str(SLOVAKIA_FILE)]).df()

temporal_check

,observed_months,expected_months,missing_months,unexpected_months
0,135,135,0,0


### 3.3 Initial Data Completeness

The completeness of the core analytical variables is assessed by counting missing values in the monthly identifier, product identifier, trading-partner identifier, trade-flow variable, and trade value.

In [7]:
core_columns = [
    "month_id",
    "product_id",
    "country_id",
    "trade_flow_name",
    "trade_value"
]

null_checks = ",\n".join(
    f'COUNT(*) FILTER (WHERE "{column}" IS NULL) AS "{column}_nulls"'
    for column in core_columns
)

quality_query = f"""
    SELECT
        COUNT(*) AS total_rows,
        {null_checks}
    FROM read_csv_auto(?)
"""

initial_quality = con.execute(
    quality_query,
    [str(SLOVAKIA_FILE)]
).df()

initial_quality

,total_rows,month_id_nulls,product_id_nulls,country_id_nulls,trade_flow_name_nulls,trade_value_nulls
0,20592924,0,0,0,0,0


### 3.4 Initial Dataset Statistics

The dataset is summarized in terms of its number of observations, distinct products, trading partners, trade flows, and monthly periods to establish a baseline for subsequent preprocessing and validation.

In [8]:
dataset_statistics = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT month_id) AS distinct_months,
        COUNT(DISTINCT product_id) AS distinct_products,
        COUNT(DISTINCT country_id) AS distinct_partners,
        COUNT(DISTINCT trade_flow_name) AS distinct_trade_flows,
        MIN(month_id) AS first_month,
        MAX(month_id) AS last_month
    FROM read_csv_auto(?)
""", [str(SLOVAKIA_FILE)]).df()

dataset_statistics

,total_rows,distinct_months,distinct_products,distinct_partners,distinct_trade_flows,first_month,last_month
0,20592924,135,5746,235,2,201501,202603


## 4. Duplicate Investigation

Duplicate observations are investigated using the proposed analytical key to distinguish repeated records from records that share the same key but contain different reported trade values.

In [9]:
duplicate_key_summary = con.execute("""
    WITH key_counts AS (
        SELECT
            month_id,
            product_id,
            country_id,
            trade_flow_name,
            COUNT(*) AS record_count
        FROM read_csv_auto(?)
        GROUP BY
            month_id,
            product_id,
            country_id,
            trade_flow_name
    )
    SELECT
        COUNT(*) FILTER (WHERE record_count > 1)
            AS duplicated_key_groups,
        COALESCE(
            SUM(record_count - 1) FILTER (WHERE record_count > 1),
            0
        ) AS excess_records,
        MAX(record_count) AS maximum_records_per_key
    FROM key_counts
""", [str(SLOVAKIA_FILE)]).df()

duplicate_key_summary

,duplicated_key_groups,excess_records,maximum_records_per_key
0,10296462,10296462.0,2


### 4.2 Exact Duplicate Records

Records identical across all seven source columns are identified to determine how many duplicated observations can be removed without discarding distinct reported values.

In [10]:
exact_duplicate_summary = con.execute("""
    WITH record_counts AS (
        SELECT
            month_id,
            product_id,
            product_name,
            country_id,
            country_name,
            trade_flow_name,
            trade_value,
            COUNT(*) AS record_count
        FROM read_csv_auto(?)
        GROUP BY
            month_id,
            product_id,
            product_name,
            country_id,
            country_name,
            trade_flow_name,
            trade_value
    )
    SELECT
        COALESCE(
            SUM(record_count - 1) FILTER (WHERE record_count > 1),
            0
        ) AS exact_duplicate_rows,
        COUNT(*) FILTER (WHERE record_count > 1)
            AS exact_duplicate_record_groups
    FROM record_counts
""", [str(SLOVAKIA_FILE)]).df()

exact_duplicate_summary

,exact_duplicate_rows,exact_duplicate_record_groups
0,10296462.0,10296462


### 4.3 Inspection of Duplicate Records

A sample of duplicated records is examined alongside their occurrence counts to verify that the identified duplicates are identical across the source columns.

In [11]:
duplicate_sample = con.execute("""
    WITH record_counts AS (
        SELECT
            month_id,
            product_id,
            product_name,
            country_id,
            country_name,
            trade_flow_name,
            trade_value,
            COUNT(*) AS record_count
        FROM read_csv_auto(?)
        GROUP BY
            month_id,
            product_id,
            product_name,
            country_id,
            country_name,
            trade_flow_name,
            trade_value
        HAVING COUNT(*) > 1
    )
    SELECT *
    FROM record_counts
    ORDER BY month_id, product_id, country_id, trade_flow_name
    LIMIT 10
""", [str(SLOVAKIA_FILE)]).df()

duplicate_sample

,month_id,product_id,product_name,country_id,country_name,trade_flow_name,trade_value,record_count
0,201501,010121,"Horses: live, pure-bred breeding animals",rus,Russia,Imports,2000,2
1,201501,010129,"Horses: live, other than pure-bred breeding an...",che,Switzerland,Exports,16000,2
2,201501,010221,"Cattle: live, pure-bred breeding animals",aut,Austria,Exports,124670,2
3,201501,010221,"Cattle: live, pure-bred breeding animals",cze,Czechia,Exports,6993,2
4,201501,010221,"Cattle: live, pure-bred breeding animals",deu,Germany,Exports,184778,2
5,201501,010221,"Cattle: live, pure-bred breeding animals",hun,Hungary,Exports,687324,2
6,201501,010221,"Cattle: live, pure-bred breeding animals",rus,Russia,Exports,1750250,2
7,201501,010229,"Cattle: live, other than pure-bred breeding an...",aut,Austria,Exports,862590,2
8,201501,010229,"Cattle: live, other than pure-bred breeding an...",cze,Czechia,Exports,95009,2
9,201501,010229,"Cattle: live, other than pure-bred breeding an...",cze,Czechia,Imports,57405,2


### 4.4 Consistency of Product and Country Names

Product and country descriptions are checked within each analytical key to ensure that the same identifiers are not associated with conflicting names.

In [12]:
name_consistency = con.execute("""
    WITH key_names AS (
        SELECT
            month_id,
            product_id,
            country_id,
            trade_flow_name,
            COUNT(DISTINCT product_name) AS product_name_count,
            COUNT(DISTINCT country_name) AS country_name_count
        FROM read_csv_auto(?)
        GROUP BY
            month_id,
            product_id,
            country_id,
            trade_flow_name
    )
    SELECT
        COUNT(*) FILTER (WHERE product_name_count > 1)
            AS conflicting_product_name_groups,
        COUNT(*) FILTER (WHERE country_name_count > 1)
            AS conflicting_country_name_groups
    FROM key_names
""", [str(SLOVAKIA_FILE)]).df()

name_consistency

,conflicting_product_name_groups,conflicting_country_name_groups
0,0,0


## 5. Data Cleaning and Consolidation

The source dataset is consolidated by removing exact duplicate records while preserving the original values of all distinct observations.

In [16]:
OUTPUT_FILE = PROCESSED_DIR / "slovakia_trade_monthly.parquet"

con.execute(f"""
    COPY (
        SELECT DISTINCT
            month_id,
            product_id,
            product_name,
            country_id,
            country_name,
            trade_flow_name,
            trade_value
        FROM read_csv_auto('{SLOVAKIA_FILE.as_posix()}')
    )
    TO '{OUTPUT_FILE.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Processed dataset saved to: {OUTPUT_FILE}")
print(f"Output size: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")

Processed dataset saved to: c:\Transfer\InterdisciplinaryProject\data\processed\slovakia_trade_monthly.parquet
Output size: 73.45 MB


## 6. Post-Processing Validation

The processed Parquet dataset is validated against the original source data to verify structural consistency, record uniqueness, temporal coverage, and preservation of aggregate trade values.

### 6.1 Structural Validation

The processed dataset is checked for its total number of observations, temporal coverage, distinct products and trading partners, missing values, and remaining duplicate analytical keys.

In [17]:
processed_validation = con.execute("""
    SELECT
        COUNT(*) AS processed_rows,
        COUNT(DISTINCT month_id) AS distinct_months,
        MIN(month_id) AS first_month,
        MAX(month_id) AS last_month,
        COUNT(DISTINCT product_id) AS distinct_products,
        COUNT(DISTINCT country_id) AS distinct_partners,
        COUNT(DISTINCT trade_flow_name) AS distinct_trade_flows,
        COUNT(*) FILTER (WHERE month_id IS NULL) AS missing_month_ids,
        COUNT(*) FILTER (WHERE product_id IS NULL) AS missing_product_ids,
        COUNT(*) FILTER (WHERE country_id IS NULL) AS missing_country_ids,
        COUNT(*) FILTER (WHERE trade_flow_name IS NULL) AS missing_trade_flows,
        COUNT(*) FILTER (WHERE trade_value IS NULL) AS missing_trade_values
    FROM read_parquet(?)
""", [str(OUTPUT_FILE)]).df()

processed_validation

,processed_rows,distinct_months,first_month,last_month,distinct_products,distinct_partners,distinct_trade_flows,missing_month_ids,missing_product_ids,missing_country_ids,missing_trade_flows,missing_trade_values
0,10296462,135,201501,202603,5746,235,2,0,0,0,0,0


### 6.2 Duplicate-Key Validation

The processed dataset is checked for repeated analytical keys to verify that exact duplicate removal has resulted in one record per month, product, trading partner, and trade flow.

In [18]:
duplicate_key_validation = con.execute("""
    SELECT COUNT(*) AS duplicated_key_groups
    FROM (
        SELECT
            month_id,
            product_id,
            country_id,
            trade_flow_name
        FROM read_parquet(?)
        GROUP BY
            month_id,
            product_id,
            country_id,
            trade_flow_name
        HAVING COUNT(*) > 1
    ) AS duplicated_keys
""", [str(OUTPUT_FILE)]).df()

duplicate_key_validation

,duplicated_key_groups
0,0


# Data Preparation and Consolidation — Slovakia

## 1. Objective

The objective of this stage was to inspect, clean, and consolidate the Slovak bilateral trade dataset into a unified monthly dataset suitable for subsequent exploratory data analysis and forecasting.

The processing workflow focused on:

* Verifying the structure and temporal coverage of the source data.
* Examining data completeness and consistency.
* Identifying and investigating duplicate observations.
* Removing exact duplicate records.
* Establishing an appropriate procedure for handling duplicate records.
* Exporting the processed dataset in Parquet format.
* Validating the resulting dataset.

The original source file was retained without modification.

## 2. Data Source and Input File

The Slovak trade data were obtained from Eurostat.

A single CSV file was used as the source dataset.

**Input file:** `trade_n_svk_m_hs.csv`

**Location:** `data/raw/trade_n_svk_m_hs.csv`

### Original schema

The source file contained the following columns:

| Variable | Description |
|---|---|
| `month_id` | Monthly observation identifier |
| `product_id` | Product classification code |
| `product_name` | Product description |
| `country_id` | Trading partner identifier |
| `country_name` | Trading partner name |
| `trade_flow_name` | Trade flow, such as imports or exports |
| `trade_value` | Reported trade value |

Unlike the South Korean dataset, the Slovak source does not contain a `weight` variable.

The `product_id` column was inferred as a string variable during schema inspection and was retained in that form.

## 3. Data Inspection and Quality Assessment

The Slovak dataset was inspected using Python and DuckDB.

DuckDB was selected for the data-processing operations because it enables analytical queries over large CSV files without requiring the entire dataset to be loaded into pandas memory.

### 3.1 Temporal coverage

The dataset was inspected for its minimum and maximum observation periods and number of distinct months.

**Results:**

* First observation: January 2015 (`201501`)
* Last observation: March 2026 (`202603`)
* Number of distinct months: 135

### 3.2 Temporal consistency check

The observed monthly periods were compared with the expected sequence of months between January 2015 and March 2026, inclusive.

| Metric | Result |
|---|---:|
| Observed months | 135 |
| Expected months | 135 |
| Missing months | 0 |
| Unexpected months | 0 |

The dataset exhibits continuous monthly coverage over the examined period.

### 3.3 Initial data completeness

The following core variables were examined for missing values:

* `month_id`
* `product_id`
* `country_id`
* `trade_flow_name`
* `trade_value`

No missing values were detected in these variables.

### 3.4 Initial dataset statistics

| Metric | Result |
|---|---:|
| Original observations | 20,592,924 |
| Distinct months | 135 |
| Distinct products | 5,746 |
| Distinct trading partners | 235 |
| Distinct trade flows | 2 |
| Missing month IDs | 0 |
| Missing product IDs | 0 |
| Missing country IDs | 0 |
| Missing trade flows | 0 |
| Missing trade values | 0 |

## 4. Duplicate Investigation

Duplicate observations were investigated using the following proposed analytical key:

`month_id + product_id + country_id + trade_flow_name`

The purpose was to determine whether repeated records represented exact repetitions or distinct observations that needed to be consolidated.

### 4.1 Duplicate-key detection

The dataset was grouped by the proposed analytical key to identify repeated key combinations.

The investigation identified:

* 10,296,462 duplicated analytical key groups.
* 10,296,462 excess records associated with duplicated keys.
* A maximum of two records per analytical key.

These results indicated that every duplicated analytical key occurred exactly twice.

### 4.2 Exact duplicate records

An exact duplicate was defined as a record identical across all seven original columns.

The investigation identified:

* 10,296,462 excess rows belonging to exact duplicate record groups.
* 10,296,462 exact duplicate record groups.

The number of exact duplicate excess rows equalled the number of excess records identified through duplicate-key detection.

Therefore, all repeated analytical keys corresponded to exact duplicate records rather than distinct trade observations.

### 4.3 Inspection of duplicate records

A sample of duplicated records was inspected to verify whether the records within each duplicated group were identical.

The inspected records occurred in pairs, with both records sharing the same values across all seven columns.

This supported the conclusion that the repeated observations could be removed without aggregating distinct trade records.

### 4.4 Consistency of product and country names

The consistency of `product_name` and `country_name` was checked within each analytical key.

| Check | Result |
|---|---:|
| Groups with conflicting product names | 0 |
| Groups with conflicting country names | 0 |

No conflicting product or country descriptions were identified within the examined groups.

### 4.5 Duplicate investigation summary

The duplicate investigation established that all repeated analytical keys corresponded to exact duplicate records, with no conflicting product or country descriptions identified.

These findings supported removing exact duplicates without aggregating distinct trade observations.

## 5. Data Cleaning and Consolidation

The processing procedure consisted of the following operations.

### Step 1: Read the source dataset

The original Slovak CSV file was read using DuckDB.

### Step 2: Remove exact duplicates

Exact duplicate rows were removed using `SELECT DISTINCT` across all seven original columns.

Since all repeated analytical keys corresponded to exact duplicate records, no aggregation of distinct trade observations was required.

### Step 3: Export the processed dataset

The resulting dataset was exported in Parquet format using Zstandard compression.

The output was stored at:

`data/processed/slovakia_trade_monthly.parquet`

The original CSV file in `data/raw/` was retained unchanged.

## 6. Post-Processing Validation

The resulting Parquet dataset was subjected to structural, completeness, temporal, and duplicate-key validation.

### 6.1 Structural validation

The processed dataset was inspected to verify its observation count, temporal coverage, and number of distinct products, trading partners, and trade flows.

| Metric | Result |
|---|---:|
| Processed observations | 10,296,462 |
| Distinct months | 135 |
| First month | 201501 |
| Last month | 202603 |
| Distinct products | 5,746 |
| Distinct trading partners | 235 |
| Distinct trade flows | 2 |

The processed dataset contains exactly half as many rows as the original dataset, consistent with the removal of the duplicated records identified during the investigation.

### 6.2 Missing-value validation

The processed dataset was checked for missing values in the core analytical columns.

| Variable | Missing values |
|---|---:|
| `month_id` | 0 |
| `product_id` | 0 |
| `country_id` | 0 |
| `trade_flow_name` | 0 |
| `trade_value` | 0 |

No missing values were detected in the checked variables.

### 6.3 Duplicate-key validation

The processed dataset was grouped by the analytical key:

`month_id + product_id + country_id + trade_flow_name`

The validation identified **zero duplicated analytical key groups**.

This confirms that the processed dataset contains no repeated combinations of month, product, trading partner, and trade flow.

## 7. Final Output

The resulting dataset contains monthly Slovak trade observations organized by product, trading partner, and trade flow.

**Output file:** `slovakia_trade_monthly.parquet`

**Format:** Parquet

**Compression:** Zstandard

**Number of observations:** 10,296,462

**Temporal coverage:** January 2015 – March 2026

The dataset has passed the structural, uniqueness, and missing-value checks conducted during this preparation stage.

## 8. Remaining Methodological Considerations

Although the consolidation and validation procedures were successful, several considerations remain relevant before the dataset is used for forecasting.

### Zero trade values

Zero trade values were not explicitly investigated during this preparation stage. Their frequency, interpretation, and potential implications for forecasting metrics will be examined during EDA and subsequent model preparation.

### Forecasting series completeness

The dataset contains 135 distinct months overall. This does not imply that every product–partner–trade-flow combination has an observation for every month.

Missing periods within individual forecasting series will therefore require separate investigation.


## 9. Conclusion

The Slovak bilateral trade dataset was successfully cleaned and consolidated into a single processed dataset.

The duplicate investigation established that all repeated analytical keys corresponded to exact duplicate records. Following their removal, the processed dataset contained no duplicate analytical keys, no missing values in the checked core columns, and continuous monthly coverage from January 2015 through March 2026.

The consolidated dataset is ready for the next stage of the project: applying the corresponding data preparation procedure to the US trade dataset, followed by cross-country consistency validation before comparative exploratory data analysis.